In [14]:
import pandas as pd
from collections import defaultdict

print("Initializing high-speed graph structures...")

maol = pd.read_csv("maol_1.1_edge_list.csv")
fafb = pd.read_csv("fafb_783_edge_list.csv")
mcns = pd.read_csv("mcns_0.9_edge_list.csv")

# 1. Build Edge Validation Sets (Hash sets for near-instantaneous O(1) checks)
# Assumes your raw dataframes are named: maol, fafb, mcns
maol_edges = set(zip(maol['source neuron id'], maol['target neuron id']))
fafb_edges = set(zip(fafb['source neuron id'], fafb['target neuron id']))
mcns_edges = set(zip(mcns['source neuron id'], mcns['target neuron id']))

print(f"Edge Sets Built -> MAOL: {len(maol_edges)}, FAFB: {len(fafb_edges)}, MCNS: {len(mcns_edges)}")

# 2. Build Bi-directional Adjacency Maps (For fast neighborhood crawling)
# We map each node to both its downstream targets (out) and upstream sources (in)
maol_adj = defaultdict(lambda: {'out': [], 'in': []})
fafb_adj = defaultdict(lambda: {'out': [], 'in': []})
mcns_adj = defaultdict(lambda: {'out': [], 'in': []})

# Populate MAOL
for src, tgt in maol_edges:
    maol_adj[src]['out'].append(tgt)
    maol_adj[tgt]['in'].append(src)

# Populate FAFB
for src, tgt in fafb_edges:
    fafb_adj[src]['out'].append(tgt)
    fafb_adj[tgt]['in'].append(src)

# Populate MCNS
for src, tgt in mcns_edges:
    mcns_adj[src]['out'].append(tgt)
    mcns_adj[tgt]['in'].append(src)

print(f"Adjacency Maps Built -> MAOL Nodes: {len(maol_adj)}, FAFB Nodes: {len(fafb_adj)}, MCNS Nodes: {len(mcns_adj)}")
print("\n✅ Cell 1 Complete! Ready for the seed selection cell.")

Initializing high-speed graph structures...
Edge Sets Built -> MAOL: 6484936, FAFB: 3732460, MCNS: 6239112
Adjacency Maps Built -> MAOL Nodes: 51669, FAFB Nodes: 138584, MCNS Nodes: 165820

✅ Cell 1 Complete! Ready for the seed selection cell.


In [15]:
import networkx as nx
print("Calculating total degrees and selecting top seeds...")

# 1. Compute total degree (In + Out) for every node using our Adjacency Maps
maol_total_deg = {node: len(data['out']) + len(data['in']) for node, data in maol_adj.items()}
fafb_total_deg = {node: len(data['out']) + len(data['in']) for node, data in fafb_adj.items()}
mcns_total_deg = {node: len(data['out']) + len(data['in']) for node, data in mcns_adj.items()}


def get_kcore_seed(adj, edge_set, k_target=None):
    G = nx.DiGraph()
    for (src, tgt) in edge_set:
        G.add_edge(src, tgt)
    # Use undirected degree for k-core (weak connectivity analog)
    G_undirected = G.to_undirected()
    core_numbers = nx.core_number(G_undirected)
    max_k = max(core_numbers.values())
    target_k = k_target or max_k
    
    # Get nodes in the innermost core
    core_nodes = [n for n, k in core_numbers.items() if k >= target_k]
    
    # Among these, find relay nodes: balanced in/out degree, not super-hubs
    candidates = []
    for n in core_nodes:
        in_d = len(adj[n]['in'])
        out_d = len(adj[n]['out'])
        total = in_d + out_d
        balance = min(in_d, out_d) / max(in_d, out_d) if max(in_d, out_d) > 0 else 0
        if 20 < total < 500:  # moderate degree
            candidates.append((n, total, balance))
    
    # Sort by degree-balance product: high total degree but not lopsided
    candidates.sort(key=lambda x: x[1] * x[2], reverse=True)
    return candidates[0][0] if candidates else None

# # 2. Sort the nodes by total degree in descending order
# maol_sorted = sorted(maol_total_deg.items(), key=lambda x: x[1], reverse=True)
# fafb_sorted = sorted(fafb_total_deg.items(), key=lambda x: x[1], reverse=True)
# mcns_sorted = sorted(mcns_total_deg.items(), key=lambda x: x[1], reverse=True)

# # 3. Extract the absolute top node from each dataset to form our initial seed triplet
# seed_m = maol_sorted[0][0]
# seed_f = fafb_sorted[0][0]
# seed_c = mcns_sorted[0][0]
seed_m = get_kcore_seed(maol_adj, maol_total_deg.items())
seed_f = get_kcore_seed(fafb_adj, fafb_total_deg.items())
seed_c = get_kcore_seed(mcns_adj, mcns_total_deg.items())

print("--- Top Ranked Hubs Found ---")
print(f"🥇 MAOL Seed Node: {seed_m} (Total Connections: {maol_total_deg[seed_m]})")
print(f"🥇 FAFB Seed Node: {seed_f} (Total Connections: {fafb_total_deg[seed_f]})")
print(f"🥇 MCNS Seed Node: {seed_c} (Total Connections: {mcns_total_deg[seed_c]})")
print("-----------------------------")
print(f"🌱 Initial Seed Triplet Locked: ({seed_m}, {seed_f}, {seed_c})")

print("\n✅ Cell 2 Complete! Ready to set up the expansion state tracker.")

Calculating total degrees and selecting top seeds...
--- Top Ranked Hubs Found ---
🥇 MAOL Seed Node: 45538 (Total Connections: 499)
🥇 FAFB Seed Node: 720575940626510169 (Total Connections: 495)
🥇 MCNS Seed Node: 11502 (Total Connections: 497)
-----------------------------
🌱 Initial Seed Triplet Locked: (45538, 720575940626510169, 11502)

✅ Cell 2 Complete! Ready to set up the expansion state tracker.


In [16]:
from collections import deque

print("Initializing expansion state with strict seed constraints...")

# 1. Start with ONLY the verified seed triplet
aligned_triplets = [(seed_m, seed_f, seed_c)]
locked_m = {seed_m}
locked_f = {seed_f}
locked_c = {seed_c}

# 2. The frontier queue starts with just this one verified anchor point
frontier_queue = deque([(seed_m, seed_f, seed_c)])

print("--- State Tracking Initialized ---")
print(f"🔒 Locked Neurons -> MAOL: {len(locked_m)}, FAFB: {len(locked_f)}, MCNS: {len(locked_c)}")
print(f"🌊 Initial Frontier Queue Size: {len(frontier_queue)} (Pure Seed Anchor)")
print("----------------------------------")
print("\n✅ Cell 3 Complete! Ready for the verified main loop.")

Initializing expansion state with strict seed constraints...
--- State Tracking Initialized ---
🔒 Locked Neurons -> MAOL: 1, FAFB: 1, MCNS: 1
🌊 Initial Frontier Queue Size: 1 (Pure Seed Anchor)
----------------------------------

✅ Cell 3 Complete! Ready for the verified main loop.


In [17]:
import time

print("🚀 Launching the Exact Topological Expansion Engine...")
start_time = time.time()
old_interval = start_time

# --- Execution Parameters ---
TARGET_SIZE = 500          
FANOUT_GUARDRAIL = 250000   
PROGRESS_INTERVAL = 25     

nodes_added = 0

while frontier_queue and len(aligned_triplets) < TARGET_SIZE:
    m_curr, f_curr, c_curr = frontier_queue.popleft()
    
    for direction in ['out', 'in']:
        m_avail = [n for n in maol_adj[m_curr][direction] if n not in locked_m]
        f_avail = [n for n in fafb_adj[f_curr][direction] if n not in locked_f] # Bug fix: was checked against locked_c accidentally
        c_avail = [n for n in mcns_adj[c_curr][direction] if n not in locked_c]
        
        if not m_avail or not f_avail or not c_avail:
            continue
            
        # 🌟 THE CORRECT STEP 1 HANDLING: 
        # If it's the very first node, skip the guardrail check entirely to allow exploration.
        # For all subsequent nodes, enforce the safety guardrail.
        if len(aligned_triplets) > 1:
            if len(m_avail) * len(f_avail) * len(c_avail) > FANOUT_GUARDRAIL:
                continue
        else:
            # Caps Step 1 search space to prevent absolute lockups on the super-hub
            m_avail = m_avail[:50]
            f_avail = f_avail[:50]
            c_avail = c_avail[:50]
            
        # Clean explicit loops
        for m_next in m_avail:
            for f_next in f_avail:
                for c_next in c_avail:
                    
                    if (m_next in locked_m) or (f_next in locked_f) or (c_next in locked_c):
                        continue

                    # 1. LOCAL GATEKEEPER (O(1) cost)
                    has_m = (m_curr, m_next) in maol_edges
                    has_f = (f_curr, f_next) in fafb_edges
                    has_c = (c_curr, c_next) in mcns_edges

                    if not (has_m == has_f == has_c):
                        continue # Drop it instantly! Skip the expensive loop below.
                        
                    # GLOBAL ISOMORPHISM VERIFICATION
                    isomorphic = True
                    for m_old, f_old, c_old in aligned_triplets:
                        if ((m_old, m_next) in maol_edges) != ((f_old, f_next) in fafb_edges) or \
                           ((f_old, f_next) in fafb_edges) != ((c_old, c_next) in mcns_edges):
                            isomorphic = False
                            break
                        if ((m_next, m_old) in maol_edges) != ((f_next, f_old) in fafb_edges) or \
                           ((f_next, f_old) in fafb_edges) != ((c_next, c_old) in mcns_edges):
                            isomorphic = False
                            break
                            
                    if isomorphic:
                        aligned_triplets.append((m_next, f_next, c_next))
                        locked_m.add(m_next)
                        locked_f.add(f_next)
                        locked_c.add(c_next)
                        frontier_queue.append((m_next, f_next, c_next))
                        
                        nodes_added += 1
                        if nodes_added % PROGRESS_INTERVAL == 0:
                            interval = time.time()
                            print(f"📈 Circuit growing... Total Aligned Nodes: {len(aligned_triplets)} / {TARGET_SIZE} ({interval - old_interval}s)")
                            old_interval = interval
                            
                        if len(aligned_triplets) >= TARGET_SIZE: break
                if len(aligned_triplets) >= TARGET_SIZE: break
            if len(aligned_triplets) >= TARGET_SIZE: break

duration = time.time() - start_time
print(f"\n🏁 COMPLETED! Runtime: {duration:.2f} seconds | Final Size: {len(aligned_triplets)} nodes")

🚀 Launching the Exact Topological Expansion Engine...
📈 Circuit growing... Total Aligned Nodes: 26 / 500 (0.28547215461730957s)
📈 Circuit growing... Total Aligned Nodes: 51 / 500 (1.7637507915496826s)
📈 Circuit growing... Total Aligned Nodes: 76 / 500 (1.8309130668640137s)
📈 Circuit growing... Total Aligned Nodes: 101 / 500 (5.377338171005249s)
📈 Circuit growing... Total Aligned Nodes: 126 / 500 (5.3452067375183105s)
📈 Circuit growing... Total Aligned Nodes: 151 / 500 (5.345001935958862s)
📈 Circuit growing... Total Aligned Nodes: 176 / 500 (9.644983053207397s)
📈 Circuit growing... Total Aligned Nodes: 201 / 500 (6.709214925765991s)
📈 Circuit growing... Total Aligned Nodes: 226 / 500 (8.19665813446045s)
📈 Circuit growing... Total Aligned Nodes: 251 / 500 (9.699788808822632s)
📈 Circuit growing... Total Aligned Nodes: 276 / 500 (18.664645195007324s)
📈 Circuit growing... Total Aligned Nodes: 301 / 500 (17.800925970077515s)
📈 Circuit growing... Total Aligned Nodes: 326 / 500 (8.050060987472

In [18]:
print("🔬 Running strict structural isomorphism audit on the generated subgraph...")

# 1. Create a clean mapping of aligned node indices for cross-dataset lookup
# This lets us translate "Position X in our circuit" to the specific ID in each brain
m_nodes = [t[0] for t in aligned_triplets]
f_nodes = [t[1] for t in aligned_triplets]
c_nodes = [t[2] for t in aligned_triplets]

num_nodes = len(aligned_triplets)
edge_counts = {'MAOL': 0, 'FAFB': 0, 'MCNS': 0}
violations = 0

# 2. Comprehensive Pairwise Matrix Audit
# We check the directed relationship between every single pair of nodes in the circuit (N^2 check)
for i in range(num_nodes):
    for j in range(num_nodes):
        if i == j:
            continue  # Skip self-connections
            
        # Check edge existence in MAOL
        has_maol = (m_nodes[i], m_nodes[j]) in maol_edges
        if has_maol: edge_counts['MAOL'] += 1
            
        # Check edge existence in FAFB
        has_fafb = (f_nodes[i], f_nodes[j]) in fafb_edges
        if has_fafb: edge_counts['FAFB'] += 1
            
        # Check edge existence in MCNS
        has_mcns = (c_nodes[i], c_nodes[j]) in mcns_edges
        if has_mcns: edge_counts['MCNS'] += 1
            
        # CRITICAL TEST: The edge state MUST be identical across all three datasets
        # They must either all be True (edge exists) or all be False (no edge exists)
        if not (has_maol == has_fafb == has_mcns):
            violations += 1
            if violations <= 5:  # Print the first few violations if they exist for debugging
                print(f"❌ Structural Mismatch Found at matrix position ({i} -> {j}):")
                print(f"   MAOL Edge: {has_maol} | FAFB Edge: {has_fafb} | MCNS Edge: {has_mcns}")

# --- Audit Report Summary ---
print("\n------------------------------------------------")
print("📊 SUBGRAPH ISOMORPHISM AUDIT REPORT")
print("------------------------------------------------")
print(f"Total Circuit Nodes Evaluated : {num_nodes}")
print(f"Total Pairwise Paths Checked  : {num_nodes * (num_nodes - 1)}")
print(f"Reconstructed Subgraph Edges  : {edge_counts}")
print(f"Total Structural Violations   : {violations}")
print("------------------------------------------------")

if violations == 0:
    print("🏆 SUCCESS: The subgraphs are 100% isomorphic across all three datasets!")
    print("🎯 Every single directed arrow is perfectly mirrored. The math is flawless.")
else:
    print("⚠️ WARNING: Isomorphism violations detected. Check the loop logic.")
print("------------------------------------------------")

print("\n✅ Cell 5 Complete! Ready to format and save the final submission CSV.")

🔬 Running strict structural isomorphism audit on the generated subgraph...

------------------------------------------------
📊 SUBGRAPH ISOMORPHISM AUDIT REPORT
------------------------------------------------
Total Circuit Nodes Evaluated : 500
Total Pairwise Paths Checked  : 249500
Reconstructed Subgraph Edges  : {'MAOL': 524, 'FAFB': 524, 'MCNS': 524}
Total Structural Violations   : 0
------------------------------------------------
🏆 SUCCESS: The subgraphs are 100% isomorphic across all three datasets!
🎯 Every single directed arrow is perfectly mirrored. The math is flawless.
------------------------------------------------

✅ Cell 5 Complete! Ready to format and save the final submission CSV.


In [19]:
import pandas as pd
import gravis as gv
import networkx as nx

print("💾 Step 1: Exporting aligned triplets to final submission CSV...")

# Create the formatted dataframe matching the required alignment structure
submission_df = pd.DataFrame(aligned_triplets, columns=['maol_id', 'fafb_id', 'mcns_id'])

# Save to disk
submission_file = "network.csv"
submission_df.to_csv(submission_file, index=False)
submission_df['fafb_id'].to_clipboard(index=False)
print(f"🏆 Submission file successfully saved to: {submission_file} ({len(submission_df)} rows)")

💾 Step 1: Exporting aligned triplets to final submission CSV...
🏆 Submission file successfully saved to: network.csv (500 rows)


In [20]:
print("\n🎨 Step 2: Constructing NetworkX graph for MAOL dataset visualization...")

# Initialize a directed NetworkX graph
G = nx.DiGraph()

# Add our 1,000 aligned MAOL nodes
m_nodes = [t[0] for t in aligned_triplets]
G.add_nodes_from(m_nodes)

# Extract only the edges that exist between these 1,000 nodes in MAOL
for i in range(len(m_nodes)):
    for j in range(len(m_nodes)):
        if i != j and (m_nodes[i], m_nodes[j]) in maol_edges:
            G.add_edge(m_nodes[i], m_nodes[j])

# Configure visual aesthetics for Gravis
for node in G.nodes:
    G.nodes[node]['size'] = 15
    G.nodes[node]['color'] = '#1f77b4'
    # Make the original seed stand out visually
    if node == seed_m:
        G.nodes[node]['size'] = 35
        G.nodes[node]['color'] = '#d62728'
        G.nodes[node]['label'] = "SEED HUB"

for u, v in G.edges:
    G.edges[u, v]['color'] = '#aaaaaa'
    G.edges[u, v]['size'] = 1.5

print(f"Graph loaded into memory: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges.")
print("Rendering interactive plot (this will spin up an embedded frame)...")

# Render using Gravis's d3 interactive engine
fig = gv.d3(G, 
            # use_node_size_attribute=True, 
            # use_node_color_attribute=True,
            # use_edge_color_attribute=True,
            # use_edge_size_attribute=True,
            # edge_curvature=0.2,
            # zoom_factor=0.5,
            # layout_algorithm_active=True
            )

# Display the interactive plot directly in the notebook
fig.display()


🎨 Step 2: Constructing NetworkX graph for MAOL dataset visualization...
Graph loaded into memory: 500 nodes, 524 edges.
Rendering interactive plot (this will spin up an embedded frame)...


In [21]:
print("\n🎨 Step 2: Constructing NetworkX graph for FAFB dataset visualization...")

# Initialize a directed NetworkX graph
G = nx.DiGraph()

# 🌟 FIX 1: Extract FAFB IDs (Index 1) from the aligned triplets
f_nodes = [t[1] for t in aligned_triplets]
G.add_nodes_from(f_nodes)

# 🌟 FIX 2: Extract edges using the correct FAFB nodes array
for i in range(len(f_nodes)):
    for j in range(len(f_nodes)):
        if i != j and (f_nodes[i], f_nodes[j]) in fafb_edges:
            G.add_edge(f_nodes[i], f_nodes[j])

# Configure visual aesthetics for Gravis
for node in G.nodes:
    G.nodes[node]['size'] = 15
    G.nodes[node]['color'] = '#1f77b4'
    
    # 🌟 FIX 3: Match against the FAFB seed variable
    if node == seed_f:
        G.nodes[node]['size'] = 35
        G.nodes[node]['color'] = '#d62728'
        G.nodes[node]['label'] = "FAFB SEED HUB"

for u, v in G.edges:
    G.edges[u, v]['color'] = '#aaaaaa'
    G.edges[u, v]['size'] = 1.5

print(f"Graph loaded into memory: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges.")
print("Rendering interactive plot...")

# Re-enable the layout properties so it mirrors the MAOL structure beautifully
fig = gv.d3(G, 
            # use_node_size_attribute=True, 
            # use_node_color_attribute=True,
            # use_edge_color_attribute=True,
            # use_edge_size_attribute=True,
            # edge_curvature=0.2,
            # zoom_factor=0.5,
            # layout_algorithm_active=True
            )

fig.display()


🎨 Step 2: Constructing NetworkX graph for FAFB dataset visualization...
Graph loaded into memory: 500 nodes, 524 edges.
Rendering interactive plot...


In [24]:
print("\n🎨 Step 2: Constructing NetworkX graph for MCNS dataset visualization...")

# Initialize a directed NetworkX graph
G = nx.DiGraph()

# 🌟 FIX 1: Extract MCNS IDs (Index 1) from the aligned triplets
c_nodes = [t[2] for t in aligned_triplets]
G.add_nodes_from(c_nodes)

# 🌟 FIX 2: Extract edges using the correct FAFB nodes array
for i in range(len(c_nodes)):
    for j in range(len(c_nodes)):
        if i != j and (c_nodes[i], c_nodes[j]) in mcns_edges:
            G.add_edge(c_nodes[i], c_nodes[j])

# Configure visual aesthetics for Gravis
for node in G.nodes:
    G.nodes[node]['size'] = 15
    G.nodes[node]['color'] = '#1f77b4'
    
    # 🌟 FIX 3: Match against the FAFB seed variable
    if node == seed_c:
        G.nodes[node]['size'] = 35
        G.nodes[node]['color'] = '#d62728'
        G.nodes[node]['label'] = "FAFB SEED HUB"

for u, v in G.edges:
    G.edges[u, v]['color'] = '#aaaaaa'
    G.edges[u, v]['size'] = 1.5

print(f"Graph loaded into memory: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges.")
print("Rendering interactive plot...")

# Re-enable the layout properties so it mirrors the MAOL structure beautifully
fig = gv.d3(G, 
            # use_node_size_attribute=True, 
            # use_node_color_attribute=True,
            # use_edge_color_attribute=True,
            # use_edge_size_attribute=True,
            # edge_curvature=0.2,
            # zoom_factor=0.5,
            # layout_algorithm_active=True
            )

fig.display()


🎨 Step 2: Constructing NetworkX graph for MCNS dataset visualization...
Graph loaded into memory: 500 nodes, 524 edges.
Rendering interactive plot...
